# K-IFRS 회계 분개 스킬 — A/B 비교 테스트

두 가지 에이전트 방식을 동일한 테스트 케이스로 비교합니다.

| | 방식 A (Sandbox) | 방식 B (Prompt-only) |
|---|---|---|
| **Backend** | `LocalShellBackend` | `FilesystemBackend` |
| **코드 실행** | ✅ `execute` 도구 사용 가능 | ❌ 없음 |
| **분개 생성** | Python 코드 작성→실행 | LLM이 규칙 기반 직접 추론 |
| **스킬 파일** | 동일 (마크다운 3개) | 동일 (마크다운 3개) |
| **장점** | 결정론적, 검증 자동화 | 배포 간단, 유연 |
| **단점** | sandbox 필요, 배포 복잡 | 비결정론적, 간혹 계정 오류 |

> 두 에이전트 모두 **동일한 스킬 파일**(SKILL.md + 참조 마크다운)을 사용합니다.  
> 차이는 에이전트가 분개를 생성하는 **방법**뿐입니다.

## 0. 환경설정

In [2]:
from Agent.config import LLM, SMALL_LLM, SMART_LLM, CLIENT

In [13]:
import os

# 스킬 파일 경로 — 실제 환경에 맞게 수정하세요
SKILL_ROOT = os.path.expanduser("./skills")

# 스킬 파일 존재 확인
skill_md = os.path.join(SKILL_ROOT, "account-by-python", "SKILL.md")
assert os.path.exists(skill_md), f"SKILL.md not found: {skill_md}"
print(f"✅ 스킬 루트: {SKILL_ROOT}")
print(f"✅ SKILL.md 확인됨")

# 하위 파일 목록
for root, dirs, files in os.walk(os.path.join(SKILL_ROOT, "k-ifrs-accounting")):
    for f in files:
        rel = os.path.relpath(os.path.join(root, f), SKILL_ROOT)
        print(f"   📄 {rel}")

✅ 스킬 루트: ./skills
✅ SKILL.md 확인됨


## 1. 에이전트 생성

두 에이전트를 각각 생성합니다. 동일한 스킬, 다른 backend + system_prompt.

In [15]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend, LocalShellBackend

# ── 공통 설정 ──
MODEL = SMALL_LLM  # 비용 최적화: 테스트 시 haiku로 변경 가능
# MODEL = "claude-haiku-4-5-20251001"

SKILLS_PATH = "/"  # backend root_dir 기준 상대 경로

### 방식 A: Sandbox 에이전트 (코드 실행 가능)

In [ ]:
# SYSTEM_PROMPT_A = """당신은 K-IFRS 회계 분개 전문가입니다.

# 사용자가 거래를 설명하면:
# 1. 스킬 파일(SKILL.md, references/*.md)을 읽어 계정과목과 패턴을 파악합니다
# 2. Python 코드를 작성하여 분개를 생성하고, execute 도구로 실행합니다
# 3. 코드에서 대차균형 검증까지 수행합니다
# 4. 실행 결과를 사용자에게 보여줍니다

# 반드시 한국어로 답변하세요.
# 금액은 천단위 콤마를 사용합니다.
# """

# LocalShellBackend: 파일 읽기 + 셸 실행(Python) 가능
# backend_a = LocalShellBackend(
#     root_dir=SKILL_ROOT,
#     virtual_mode=True,
#     inherit_env=True,  # ANTHROPIC_API_KEY 등 환경변수 상속
# )

# agent_a = create_deep_agent(
#     model=MODEL,
#     # system_prompt=SYSTEM_PROMPT_A,
#     skills=[SKILLS_PATH],
#     backend=backend_a,
#     name="sandbox-agent",
# )

# print("✅ 방식 A 에이전트 생성 완료 (Sandbox)")

✅ 방식 A 에이전트 생성 완료 (Sandbox)


In [ ]:
# from langchain_core.messages import HumanMessage


# result = agent_a.invoke({
#     "messages": [HumanMessage(content='(account-by-python skill을 이용하시오) A사에 상품 2,000만원을 외상으로 판매했다. 부가세 10% 별도.')]
# })

Exception in thread Thread-29 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\16Z90P\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\16Z90P\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\16Z90P\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0xec in position 7: illegal multibyte sequence
Exception in thread Thread-31 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\16Z90P\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\16Z90P\AppData\Roaming\uv\python\cpython-3.12.12-windows-x

KeyboardInterrupt: 

In [11]:
result

{'messages': [HumanMessage(content='(account-by-python skill을 이용하시오) A사에 상품 2,000만원을 외상으로 판매했다. 부가세 10% 별도.', additional_kwargs={}, response_metadata={}, id='7183abb1-1248-4483-9e70-eeb22d5c41f7'),
  AIMessage(content=[{'text': 'account-by-python 스킬을 확인하겠습니다.', 'type': 'text'}, {'id': 'toolu_01J8nPjKUnfhHAJ5BQpqF7V1', 'caller': {'type': 'direct'}, 'input': {'file_path': '/skills/account-by-python'}, 'name': 'read_file', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01Jy6z4fgm5boGj1q16nT8FA', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 7823, 'inference_geo': 'not_available', 'input_tokens': 3, 'output_tokens': 83, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, name

### 방식 B: Prompt-only 에이전트 (코드 실행 불가)

In [19]:
SYSTEM_PROMPT_B = """당신은 K-IFRS 회계 분개 전문가입니다.

사용자가 거래를 설명하면:
1. 스킬 파일(SKILL.md, references/*.md)을 읽어 계정과목과 분개 패턴을 파악합니다
2. 매칭되는 패턴이 있으면 해당 패턴을 적용합니다
3. 없으면 SKILL.md의 "분개 추론 규칙"에 따라 직접 추론합니다
4. "검증 체크리스트" 4단계를 모두 수행합니다
5. 지정된 출력 형식으로 분개를 제시합니다

코드 실행은 사용하지 마세요. 스킬 문서의 규칙에 따라 직접 추론하세요.
반드시 한국어로 답변하세요.
금액은 천단위 콤마를 사용합니다.
"""

# FilesystemBackend: 파일 읽기만 가능 (코드 실행 불가)
backend_b = FilesystemBackend(
    root_dir=SKILL_ROOT,
    virtual_mode=True,
)

agent_b = create_deep_agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT_B,
    skills=[SKILLS_PATH],
    backend=backend_b,
    name="prompt-agent",
)

print("✅ 방식 B 에이전트 생성 완료 (Prompt-only)")

✅ 방식 B 에이전트 생성 완료 (Prompt-only)


In [20]:
from langchain_core.messages import HumanMessage


result = agent_b.invoke({
    "messages": [HumanMessage(content='현재 가용한 Skill 목록과 설명을 부탁해')]
})

In [21]:
result

{'messages': [HumanMessage(content='현재 가용한 Skill 목록과 설명을 부탁해', additional_kwargs={}, response_metadata={}, id='16fd75a6-ed4a-42fb-95d2-78cff4fb3511'),
  AIMessage(content='현재 가용한 Skill은 다음과 같습니다:\n\n## 📚 Available Skills\n\n### **account-by-md** (K-IFRS 회계 분개 전문가)\n- **경로**: `/account-by-md/SKILL.md`\n- **설명**: K-IFRS(한국채택국제회계기준) 기반 복식부기 분개(Journal Entry) 생성 및 검증 스킬\n- **주요 기능**:\n  - 거래 설명 → 분개 자동 생성\n  - 계정과목 매칭 및 분개 패턴 적용\n  - 검증 체크리스트 4단계 수행\n  - 한국어 기반 회계 처리\n\n---\n\n이 Skill을 사용하려면 거래 내용을 설명해주시면, 저는 자동으로 SKILL.md를 읽고 K-IFRS 기준에 따라 분개를 생성해드립니다.\n\n예시: *"외상매출금 500만원을 현금으로 회수했어"* → 분개 자동 생성\n\n필요하신 작업이 있으신가요?', additional_kwargs={}, response_metadata={'id': 'msg_01P5sh5Bi5F3hKuPsST2VRLD', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 7236}, 'cache_creation_input_tokens': 7236, 'cache_read_input_tokens': 0, 'inference_geo': 'not_availa

In [22]:
from langchain_core.messages import HumanMessage


result = agent_b.invoke({
    "messages": [HumanMessage(content='A사에 상품 2,000만원을 외상으로 판매했다. 부가세 10% 별도. 이 거래 회계처리해봐')]
})

In [23]:
ai_messages = [m for m in result["messages"] if m.type == "ai" and m.content]
if ai_messages:
    answer = ai_messages[-1].content
    # content가 list인 경우 (멀티모달) 텍스트만 추출
    if isinstance(answer, list):
        answer = "\n".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in answer
        )
    print(answer)

완벽합니다. 이제 거래를 분석하고 분개를 생성하겠습니다.

## 거래 분석

**거래 내용:**
- A사에 상품 2,000만원을 외상으로 판매
- 부가세 10% 별도

**금액 계산:**
- 상품 판매액: 20,000,000원
- 부가세(10%): 2,000,000원
- 매출채권 총액: 22,000,000원

## 패턴 매칭

`transaction_patterns.md`의 **SALE_001: 외상매출 (부가세 포함)** 패턴과 정확히 일치합니다.

**패턴 구조:**
| 구분 | 계정과목 | 금액 |
|------|---------|------|
| (차) | 매출채권 | {amount} + {vat} |
| (대) | 수익(매출액) | {amount} |
| (대) | 부가세예수금 | {vat} |

## 분개 생성

```
[분개] A사에 상품 2,000만원을 외상으로 판매 (부가세 10% 별도)
  패턴: SALE_001 (외상매출 - 부가세 포함)
──────────────────────────────────────────
  구분     계정과목                    금액
──────────────────────────────────────────
  (차)   매출채권                22,000,000
    (대)   수익(매출액)           20,000,000
    (대)   부가세예수금            2,000,000
──────────────────────────────────────────
  차변 합계                     22,000,000
  대변 합계                     22,000,000
  → ✅ 대차 균형
```

## 검증 체크리스트

**Level 1: 대차 균형** ✅
- ✅ 차변 합계(22,000,000) = 대변 합계(22,000,000)
- ✅ 차변 항목 1개 이상 (매출채권)
- ✅ 대변 항목 2개 이상 (수익, 부가세예수금)
- ✅ 모든 금액

## 3. 테스트 실행

### 테스트 케이스 설계 원칙
- **TC1**: 단순 패턴 매칭 (외상매출) → 두 방식 모두 정확해야 함
- **TC2**: 복합 거래 (유형자산 처분) → 4줄 분개, 계산 필요
- **TC3**: 패턴 외 거래 → LLM 추론 능력 비교
- **TC4**: 비즈니스 규칙 위반 감지 → 영업권 손상환입 불가 (IAS 36.124)

### TC1: 외상매출 (단순 패턴 매칭)

In [ ]:
from langchain_core.messages import HumanMessage

result = agent_b.invoke({
    "messages": [HumanMessage(content='A사에 상품 2,000만원을 외상으로 판매했다. 부가세 10% 별도. 이 거래 회계처리해봐')]
})

ai_messages = [m for m in result["messages"] if m.type == "ai" and m.content]
if ai_messages:
    answer = ai_messages[-1].content
    # content가 list인 경우 (멀티모달) 텍스트만 추출
    if isinstance(answer, list):
        answer = "\n".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in answer
        )
    print(answer)

### TC2: 유형자산 처분 (복합 계산)

In [24]:
from langchain_core.messages import HumanMessage

result = agent_b.invoke({
    "messages": [HumanMessage(content='취득원가 1억원인 기계장치(감가상각누계액 6,000만원)를 8,000만원에 현금 처분했다. 이 거래 회계처리해봐')]
})

ai_messages = [m for m in result["messages"] if m.type == "ai" and m.content]
if ai_messages:
    answer = ai_messages[-1].content
    # content가 list인 경우 (멀티모달) 텍스트만 추출
    if isinstance(answer, list):
        answer = "\n".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in answer
        )
    print(answer)

이제 거래를 분석하겠습니다.

## 거래 분석

**거래 내용:**
- 취득원가: 1억원 (100,000천원)
- 감가상각누계액: 6,000만원 (60,000천원)
- 처분대가: 8,000만원 (80,000천원)
- 장부가액: 100,000 - 60,000 = 40,000천원
- 처분손실: 40,000 - 80,000 = -40,000천원 → **처분이익 40,000천원**

## 패턴 매칭

`transaction_patterns.md`의 **PPE_003: 유형자산 처분 (이익)** 패턴과 정확히 매칭됩니다.

## 분개 구성

| 구분 | 계정과목 | 금액 |
|------|---------|------|
| (차) | 현금및현금성자산 | 80,000 |
| (차) | 유형자산 감가상각누계액 | 60,000 |
| (대) | 기계장치 | 100,000 |
| (대) | 유형자산처분이익 | 40,000 |

## 검증 체크리스트

✅ **Level 1: 대차 균형**
- 차변: 80,000 + 60,000 = 140,000
- 대변: 100,000 + 40,000 = 140,000
- ✅ 균형

✅ **Level 2: 계정과목 유효성**
- 현금및현금성자산: chart_of_accounts.md 확인 ✅
- 유형자산 감가상각누계액: chart_of_accounts.md 확인 (contra_asset) ✅
- 기계장치: chart_of_accounts.md 확인 ✅
- 유형자산처분이익: chart_of_accounts.md 확인 ✅
- Abstract 계정 없음 ✅

✅ **Level 3: Balance 방향 적정성**
- 현금및현금성자산 (debit): 차변 기입 → 증가 ✅
- 유형자산 감가상각누계액 (credit): 차변 기입 → 감소 ✅ (contra 계정이므로 차변 기입 = 감소)
- 기계장치 (debit): 대변 기입 → 감소 ✅
- 유형자산처분이익 (credit): 대변 기입 → 증가 ✅

✅ **Level 4: K-IFRS 비즈니스 규칙**
-

### TC3: 패턴 외 거래 (LLM 추론)

In [25]:
from langchain_core.messages import HumanMessage

result = agent_b.invoke({
    "messages": [HumanMessage(content='해외 자회사로부터 배당금 5억원을 현금으로 수취했다. 이 거래 회계처리해봐')]
})

ai_messages = [m for m in result["messages"] if m.type == "ai" and m.content]
if ai_messages:
    answer = ai_messages[-1].content
    # content가 list인 경우 (멀티모달) 텍스트만 추출
    if isinstance(answer, list):
        answer = "\n".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in answer
        )
    print(answer)

패턴 파일을 다 읽었습니다. 배당금 수익 관련 패턴이 명시적으로 없으므로, 직접 추론하겠습니다.

## 분석

**거래 내용:**
- 해외 자회사로부터 배당금 5억원을 현금으로 수취

**Step 1: 거래 분석**
- 현금(자산) 증가: 500,000천원
- 배당금수익(수익) 발생: 500,000천원

**Step 2: 패턴 매칭**
- 배당금 수익 관련 명시적 패턴이 없음 → 직접 추론 필요

**Step 3: 직접 추론**

chart_of_accounts.md에서 확인한 계정:
- 현금및현금성자산: Balance = debit (자산)
- 배당금수익: Balance = credit (수익)

분개 구성:
- 현금 증가 → 차변 기입 (자산의 자연적 잔액)
- 배당금수익 발생 → 대변 기입 (수익의 자연적 잔액)

**Step 4: 검증 체크리스트**

✅ **Level 1: 대차 균형**
- 차변: 500,000천원
- 대변: 500,000천원
- 차변 = 대변 ✓

✅ **Level 2: 계정과목 유효성**
- 현금및현금성자산: chart_of_accounts.md에 존재 ✓
- 배당금수익: chart_of_accounts.md에 존재 ✓
- Abstract 계정 미사용 ✓

✅ **Level 3: Balance 방향 적정성**
- 현금및현금성자산(debit) → 차변 기입 = 증가 ✓
- 배당금수익(credit) → 대변 기입 = 발생 ✓

✅ **Level 4: K-IFRS 비즈니스 규칙**
- 영업권 손상환입 금지: 해당 없음 ✓
- 소계 계정 사용 금지: 해당 없음 ✓
- 기타 규칙: 해당 없음 ✓

---

## 분개

```
[분개] 해외 자회사로부터 배당금 5억원을 현금으로 수취
  패턴: 직접 추론
──────────────────────────────────────────
  구분     계정과목                    금액
──────────────────────────────────────────
  (차)   현금및현금

### TC4: K-IFRS 비즈니스 규칙 검증

In [26]:
from langchain_core.messages import HumanMessage

result = agent_b.invoke({
    "messages": [HumanMessage(content='전기에 인식한 영업권 손상차손 3억원을 당기에 환입 처리하고 싶다. 이 거래 회계처리해봐')]
})

ai_messages = [m for m in result["messages"] if m.type == "ai" and m.content]
if ai_messages:
    answer = ai_messages[-1].content
    # content가 list인 경우 (멀티모달) 텍스트만 추출
    if isinstance(answer, list):
        answer = "\n".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in answer
        )
    print(answer)

완벽합니다! 이제 거래를 분석하고 분개를 생성하겠습니다.

## 거래 분석

**거래 내용**: 전기에 인식한 영업권 손상차손 3억원을 당기에 환입 처리

**파악 사항**:
- 손상차손 환입 거래
- 금액: 300,000천원
- 대상 자산: 영업권(Goodwill)

## 패턴 매칭

`transaction_patterns.md`의 **IMPAIR_002: 손상차손 환입** 패턴을 찾았습니다.

```
### IMPAIR_002: 손상차손 환입
**키워드**: 손상환입, 손상차손환입, 회수가능액 회복

| 구분 | 계정과목 | 금액 |
|------|---------|------|
| (차) | [해당 자산 계정] | {amount} |
| (대) | 손상차손환입 | {amount} |

> ⚠️ **영업권(goodwill) 손상차손은 환입 불가** (IAS 36.124)
```

## 검증 체크리스트 - Level 4 (K-IFRS 비즈니스 규칙)

**⚠️ 중대 오류 발견**

SKILL.md의 검증 체크리스트 Level 4에 명시되어 있습니다:

> **영업권 손상환입 금지**: 영업권(goodwill)과 손상차손환입이 동시에 등장하면 **오류** (IAS 36.124)

그리고 transaction_patterns.md의 IMPAIR_002 패턴에도 명확히 기재되어 있습니다:

> ⚠️ **영업권(goodwill) 손상차손은 환입 불가** (IAS 36.124)

---

## 결론

**이 거래는 회계처리할 수 없습니다.**

### 이유

K-IFRS(IAS 36 자산손상)에 따르면, **영업권의 손상차손은 환입이 금지**되어 있습니다. 

- 영업권은 무형자산 중에서도 특수한 지위를 가지고 있습니다
- 한 번 손상차손으로 인식되면, 이후 회수가능액이 회복되더라도 환입할 수 없습니다
- 이는 영업권의 보수적 회계처리 원칙입니다

### 대안

만약 당기에 영업권 손상차손을 수정해야 한다면:
1. **전기 오류 수정**: 전기 분개를 역분개하고 올바

## 4. 커스텀 테스트

아래 셀에서 원하는 거래를 직접 입력하여 테스트할 수 있습니다.

In [27]:
from langchain_core.messages import HumanMessage

result = agent_b.invoke({
    "messages": [HumanMessage(content='1월 1일에 12개월 운용리스 계약을 체결하고 보증금 1,000만원을 지급했다. 매월 말에 리스료 100만원을 지급한다. 이 거래 시점과 월말의 회계처리해봐')]
})

ai_messages = [m for m in result["messages"] if m.type == "ai" and m.content]
if ai_messages:
    answer = ai_messages[-1].content
    # content가 list인 경우 (멀티모달) 텍스트만 추출
    if isinstance(answer, list):
        answer = "\n".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in answer
        )
    print(answer)

완벽합니다! 리스 패턴을 찾았습니다. 이제 거래를 분석하겠습니다.

## 거래 분석

**거래 내용:**
- 1월 1일: 12개월 운용리스 계약 체결, 보증금 1,000만원 지급
- 매월 말: 리스료 100만원 지급

**문제점 파악:**
사용자가 "운용리스"라고 명시했지만, K-IFRS 16(IFRS 16)에서는 **운용리스와 금융리스 구분이 폐지**되었습니다. 모든 리스는 **사용권자산과 리스부채**로 인식됩니다.

따라서 이 거래는:
1. **리스 개시 시점(1월 1일)**: 사용권자산과 리스부채 인식
2. **월말 리스료 지급**: 리스부채 원금 + 이자 지급

---

## 분개 생성

### **Step 1: 리스 개시 시점 (1월 1일)**

**리스 개시 시 인식할 금액 계산:**
- 보증금: 1,000만원 (리스 개시 시 지급하는 리스료)
- 미래 리스료: 100만원 × 11개월 = 1,100만원
- **리스부채 현재가치 계산** (할인율 미제시 시 암묵적 차입이율 사용)

할인율이 제시되지 않았으므로, 보증금과 미래 리스료를 단순합산하여 처리합니다:
- **총 리스료**: 1,000만원 + (100만원 × 12개월) = 2,200만원

**LEASE_001 패턴 적용:**

| 구분 | 계정과목 | 금액 |
|------|---------|------|
| (차) | 사용권자산 | 2,200만원 |
| (대) | 비유동 리스부채 | 2,200만원 |

---

### **Step 2: 월말 리스료 지급 (1월 31일, 2월 28일, ... 12월 31일)**

**월별 처리:**
리스료 100만원 지급 시, 이를 원금과 이자로 분리합니다.

**단순화 가정** (할인율 미제시):
- 월별 리스료: 100만원 (원금으로 처리)
- 이자: 0 (또는 별도 계산 필요)

**LEASE_002 패턴 적용:**

| 구분 | 계정과목 | 금액 |
|------|---------|------|
| (차) | 유동 리스부채 | 100만원 |
| (대) |

## 6. 비용 비교 팁

| 모델 | Input (1M tokens) | Output (1M tokens) | 용도 |
|------|-------------------|-------------------|------|
| Haiku 4.5 | $0.80 | $4.00 | 패턴 매칭, 단순 분개 |
| Sonnet 4.5 | $3.00 | $15.00 | 복합 거래, 추론 |

> 💡 **실무 팁**: 방식 B에서 `MODEL = "claude-haiku-4-5-20251001"`로 변경하면  
> 단순 패턴 매칭은 Haiku로도 충분한지 확인할 수 있습니다.  
> 방식 A는 코드 생성이 필요하므로 Sonnet 이상이 안정적입니다.